# Feature Selection Experiments

This notebook tests whether feature selection improves the volatility prediction models after adding the expanded FRED macro columns.

The goal is to compare different feature sets instead of automatically using every available feature.

# Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import time

from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression, SelectFromModel
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Paths

In [2]:
FEATURE_DATA_PATH = Path("../../data/processed/features/feature_engineered_dataset.csv")
OUTPUT_PATH = Path("../../data/processed/modeling/feature_selection")

In [3]:
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Load Data

In [4]:
df = pd.read_csv(FEATURE_DATA_PATH, parse_dates=["Date"])

In [5]:
df.shape

(41370, 30)

In [6]:
df.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_decimal,vix,treasury_10yr_pct,yield_curve_spread,is_inverted,fed_funds_rate_pct,...,rolling_return_20d,abs_return,squared_return,rolling_abs_return_20d,rolling_squared_return_20d,rolling_volatility_5d,rolling_volatility_20d,moving_avg_20d,price_to_moving_avg_20d,future_volatility_20d
0,2018-01-31,AAPL,39.138020,0.002755,0.0146,13.54,2.72,1.26,0,1.41,...,-0.001378,0.002755,0.000008,0.006855,0.000087,0.011013,0.009462,40.695438,0.961730,0.022707
1,2018-02-01,AAPL,39.219841,0.002091,0.0148,13.47,2.78,1.30,0,1.42,...,-0.001265,0.002091,0.000004,0.006951,0.000087,0.010065,0.009490,40.643427,0.964974,0.022726
2,2018-02-02,AAPL,37.518093,-0.043390,0.0148,17.31,2.84,1.36,0,1.42,...,-0.003667,0.043390,0.001883,0.008888,0.000180,0.019425,0.013250,40.496978,0.926442,0.019948
3,2018-02-05,AAPL,36.580715,-0.024985,0.0151,37.32,2.77,1.26,0,1.42,...,-0.005485,0.024985,0.000624,0.009568,0.000205,0.019936,0.013567,40.280635,0.908146,0.018715
4,2018-02-06,AAPL,38.109501,0.041792,0.0152,29.98,2.79,1.27,0,1.42,...,-0.003210,0.041792,0.001747,0.011472,0.000292,0.032292,0.017207,40.148329,0.949218,0.017050


# Define Target and Feature Groups

In [7]:
TARGET_COL = "future_volatility_20d"

In [8]:
market_features = [
    "daily_return",
    "return_lag_1",
    "return_lag_5",
    "rolling_return_5d",
    "rolling_return_20d",
    "abs_return",
    "squared_return",
    "rolling_abs_return_20d",
    "rolling_squared_return_20d",
    "rolling_volatility_5d",
    "rolling_volatility_20d",
    "price_to_moving_avg_20d",
]

In [9]:
macro_features = [
    "risk_free_rate_decimal",
    "vix",
    "treasury_10yr_pct",
    "yield_curve_spread",
    "is_inverted",
    "fed_funds_rate_pct",
    "unemployment_rate_pct",
    "recession_flag",
    "cpi_pct_change",
]

In [10]:
selected_macro_features = [
    "vix",
    "yield_curve_spread",
    "is_inverted",
    "unemployment_rate_pct",
    "cpi_pct_change",
]

In [11]:
feature_sets = {
    "market_only": market_features,
    "full_macro": market_features + macro_features,
    "selected_macro": market_features + selected_macro_features,
}

# Train/Test Split With Time Stamps

In [12]:
SPLIT_DATE = pd.Timestamp("2024-01-01")

train = df[df["Date"] < SPLIT_DATE].copy()
test = df[df["Date"] >= SPLIT_DATE].copy()

split_metadata = {
    "split_date": SPLIT_DATE.date().isoformat(),
    "train_start_date": train["Date"].min().date().isoformat(),
    "train_end_date": train["Date"].max().date().isoformat(),
    "test_start_date": test["Date"].min().date().isoformat(),
    "test_end_date": test["Date"].max().date().isoformat(),
    "train_rows": len(train),
    "test_rows": len(test),
}

In [13]:
split_metadata

{'split_date': '2024-01-01',
 'train_start_date': '2018-01-31',
 'train_end_date': '2023-12-29',
 'test_start_date': '2024-01-02',
 'test_end_date': '2025-12-01',
 'train_rows': 31269,
 'test_rows': 10101}

# Evaluation Function

In [14]:
def evaluate_model(model_name, feature_set_name, selected_features, y_true, y_pred, training_metadata):
    return {
        "model": model_name,
        "feature_set": feature_set_name,
        "num_features": len(selected_features),
        "features": ", ".join(selected_features),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        **split_metadata,
        **training_metadata,
    }

# Models to Compare

In [15]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0]))
    ]),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        max_depth=8,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    ),
    "Gradient Boosting": HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        l2_regularization=0.01,
        random_state=42
    ),
}

# Baseline Feature-Set Experiments

This section compares market-only features, full macro features, and selected macro features across all main supervised models.

In [16]:
experiment_results = []

In [17]:
for feature_set_name, features in feature_sets.items():
    clean_train = train.dropna(subset=features + [TARGET_COL]).copy()
    clean_test = test.dropna(subset=features + [TARGET_COL]).copy()

    X_train = clean_train[features]
    y_train = clean_train[TARGET_COL]

    X_test = clean_test[features]
    y_test = clean_test[TARGET_COL]

    for model_name, model in models.items():
        training_start_timestamp = pd.Timestamp.now()
        training_start_time = time.perf_counter()

        model.fit(X_train, y_train)

        training_end_time = time.perf_counter()
        training_end_timestamp = pd.Timestamp.now()

        y_pred = model.predict(X_test)

        training_metadata = {
            "training_start_timestamp": training_start_timestamp.isoformat(),
            "training_end_timestamp": training_end_timestamp.isoformat(),
            "training_duration_seconds": training_end_time - training_start_time,
        }

        result = evaluate_model(
            model_name,
            feature_set_name,
            features,
            y_test,
            y_pred,
            training_metadata
        )

        experiment_results.append(result)

In [18]:
feature_set_results = pd.DataFrame(experiment_results)
feature_set_results.sort_values("RMSE")

,model,feature_set,num_features,features,MAE,RMSE,R2,split_date,train_start_date,train_end_date,test_start_date,test_end_date,train_rows,test_rows,training_start_timestamp,training_end_timestamp,training_duration_seconds
8,Linear Regression,selected_macro,17,"daily_return, return_lag_1, return_lag_5, roll...",0.004002,0.005929,0.309563,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:27:18.358863,2026-07-25T11:27:18.367857,0.008966
9,Ridge Regression,selected_macro,17,"daily_return, return_lag_1, return_lag_5, roll...",0.004004,0.005930,0.309471,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:27:18.370337,2026-07-25T11:27:18.395778,0.025417
0,Linear Regression,market_only,12,"daily_return, return_lag_1, return_lag_5, roll...",0.004044,0.005985,0.296477,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:27:13.876752,2026-07-25T11:27:13.903517,0.026739
1,Ridge Regression,market_only,12,"daily_return, return_lag_1, return_lag_5, roll...",0.004046,0.005986,0.296300,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:27:13.906009,2026-07-25T11:27:13.927916,0.021889
7,Gradient Boosting,full_macro,21,"daily_return, return_lag_1, return_lag_5, roll...",0.004095,0.006003,0.292264,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:27:18.047949,2026-07-25T11:27:18.331032,0.283060
3,Gradient Boosting,market_only,12,"daily_return, return_lag_1, return_lag_5, roll...",0.004041,0.006086,0.272456,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:27:14.989958,2026-07-25T11:27:16.651270,1.661288
2,Random Forest,market_only,12,"daily_return, return_lag_1, return_lag_5, roll...",0.004078,0.006093,0.270822,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:27:13.930471,2026-07-25T11:27:14.962708,1.032213
6,Random Forest,full_macro,21,"daily_return, return_lag_1, return_lag_5, roll...",0.004392,0.006912,0.061734,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:27:16.722678,2026-07-25T11:27:18.020883,1.298183
10,Random Forest,selected_macro,17,"daily_return, return_lag_1, return_lag_5, roll...",0.004511,0.007288,-0.043039,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:27:18.399402,2026-07-25T11:27:19.710678,1.311248
11,Gradient Boosting,selected_macro,17,"daily_return, return_lag_1, return_lag_5, roll...",0.004681,0.007370,-0.066717,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:27:19.739318,2026-07-25T11:27:20.091864,0.352523


# SelectKBest Feature Selection

SelectKBest chooses the top features based on a statistical relationship with the target variable.

In [19]:
full_features = market_features + macro_features

In [20]:
clean_train = train.dropna(subset=full_features + [TARGET_COL]).copy()
clean_test = test.dropna(subset=full_features + [TARGET_COL]).copy()

In [21]:
X_train_full = clean_train[full_features]
y_train_full = clean_train[TARGET_COL]

In [22]:
X_test_full = clean_test[full_features]
y_test_full = clean_test[TARGET_COL]

In [23]:
kbest = SelectKBest(score_func=f_regression, k=10)
kbest.fit(X_train_full, y_train_full)

SelectKBest(score_func=<function f_regression at 0x00000257D44907C0>)

In [24]:
kbest_features = X_train_full.columns[kbest.get_support()].tolist()

In [25]:
kbest_features

['rolling_return_20d',
 'abs_return',
 'squared_return',
 'rolling_abs_return_20d',
 'rolling_squared_return_20d',
 'rolling_volatility_5d',
 'rolling_volatility_20d',
 'price_to_moving_avg_20d',
 'vix',
 'recession_flag']

# Mutual Information Feature Selection

Mutual information can capture non-linear relationships between features and the target.

In [26]:
mutual_info_scores = mutual_info_regression(
    X_train_full,
    y_train_full,
    random_state=42
)

In [27]:
mutual_info_ranking = (
    pd.DataFrame({
        "feature": full_features,
        "mutual_info_score": mutual_info_scores
    })
    .sort_values("mutual_info_score", ascending=False)
)

In [28]:
mutual_info_features = mutual_info_ranking.head(10)["feature"].tolist()

In [29]:
mutual_info_ranking

,feature,mutual_info_score
10,rolling_volatility_20d,0.547937
8,rolling_squared_return_20d,0.546436
7,rolling_abs_return_20d,0.492515
20,cpi_pct_change,0.470518
17,fed_funds_rate_pct,0.355017
9,rolling_volatility_5d,0.320278
12,risk_free_rate_decimal,0.298655
18,unemployment_rate_pct,0.223787
15,yield_curve_spread,0.180473
14,treasury_10yr_pct,0.180107


# Lasso-Based Feature Selection

Lasso can shrink weak feature coefficients toward zero. Features with non-zero coefficients are kept.

In [30]:
lasso_selector = Pipeline([
    ("scaler", StandardScaler()),
    ("selector", SelectFromModel(
        LassoCV(cv=5, random_state=42, max_iter=10000)
    ))
])

In [31]:
lasso_selector.fit(X_train_full, y_train_full)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('selector', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](21,)","['daily_return','return_lag_1','return_lag_5',...,'unemployment_rate_pct', 'recession_flag','cpi_pct_change']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,21
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [32]:
lasso_support = lasso_selector.named_steps["selector"].get_support()
lasso_features = X_train_full.columns[lasso_support].tolist()

In [33]:
lasso_features

['daily_return',
 'return_lag_1',
 'return_lag_5',
 'rolling_return_5d',
 'rolling_return_20d',
 'abs_return',
 'squared_return',
 'rolling_abs_return_20d',
 'rolling_squared_return_20d',
 'rolling_volatility_5d',
 'rolling_volatility_20d',
 'price_to_moving_avg_20d',
 'risk_free_rate_decimal',
 'vix',
 'yield_curve_spread',
 'is_inverted',
 'fed_funds_rate_pct',
 'unemployment_rate_pct',
 'recession_flag',
 'cpi_pct_change']

# Model-Based Feature Selection With Random Forest

Random Forest feature importance can select features that are most useful for tree-based prediction.

In [34]:
rf_selector_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=8,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

In [35]:
rf_selector_model.fit(X_train_full, y_train_full)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",8
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",20
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max

In [36]:
rf_feature_importance = (
    pd.DataFrame({
        "feature": full_features,
        "importance": rf_selector_model.feature_importances_
    })
    .sort_values("importance", ascending=False)
)

In [37]:
rf_selected_features = rf_feature_importance.head(10)["feature"].tolist()

In [38]:
rf_feature_importance

,feature,importance
7,rolling_abs_return_20d,0.459918
12,risk_free_rate_decimal,0.116380
20,cpi_pct_change,0.102078
13,vix,0.063705
15,yield_curve_spread,0.045468
19,recession_flag,0.038882
9,rolling_volatility_5d,0.032736
17,fed_funds_rate_pct,0.026889
10,rolling_volatility_20d,0.019842
14,treasury_10yr_pct,0.019651


# Compare Selected Feature Sets Across Models

In [39]:
selected_feature_sets = {
    "select_k_best": kbest_features,
    "mutual_info_top_10": mutual_info_features,
    "lasso_selected": lasso_features,
    "random_forest_top_10": rf_selected_features,
}

In [40]:
selected_results = []

In [41]:
for feature_set_name, features in selected_feature_sets.items():
    if len(features) == 0:
        continue

    clean_train = train.dropna(subset=features + [TARGET_COL]).copy()
    clean_test = test.dropna(subset=features + [TARGET_COL]).copy()

    X_train = clean_train[features]
    y_train = clean_train[TARGET_COL]

    X_test = clean_test[features]
    y_test = clean_test[TARGET_COL]

    for model_name, model in models.items():
        training_start_timestamp = pd.Timestamp.now()
        training_start_time = time.perf_counter()

        model.fit(X_train, y_train)

        training_end_time = time.perf_counter()
        training_end_timestamp = pd.Timestamp.now()

        y_pred = model.predict(X_test)

        training_metadata = {
            "training_start_timestamp": training_start_timestamp.isoformat(),
            "training_end_timestamp": training_end_timestamp.isoformat(),
            "training_duration_seconds": training_end_time - training_start_time,
        }

        result = evaluate_model(
            model_name,
            feature_set_name,
            features,
            y_test,
            y_pred,
            training_metadata
        )

        selected_results.append(result)

In [42]:
selected_feature_results = pd.DataFrame(selected_results)
selected_feature_results.sort_values("RMSE")

,model,feature_set,num_features,features,MAE,RMSE,R2,split_date,train_start_date,train_end_date,test_start_date,test_end_date,train_rows,test_rows,training_start_timestamp,training_end_timestamp,training_duration_seconds
0,Linear Regression,select_k_best,10,"rolling_return_20d, abs_return, squared_return...",0.003992,0.005971,0.299858,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:35:07.344614,2026-07-25T11:35:07.349058,0.004415
1,Ridge Regression,select_k_best,10,"rolling_return_20d, abs_return, squared_return...",0.003994,0.005971,0.299840,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:35:07.350981,2026-07-25T11:35:07.362160,0.011162
2,Random Forest,select_k_best,10,"rolling_return_20d, abs_return, squared_return...",0.004017,0.005971,0.299816,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:35:07.364282,2026-07-25T11:35:08.071649,0.707354
7,Gradient Boosting,mutual_info_top_10,10,"rolling_volatility_20d, rolling_squared_return...",0.004192,0.006074,0.275508,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:35:08.882660,2026-07-25T11:35:09.066777,0.184096
11,Gradient Boosting,lasso_selected,20,"daily_return, return_lag_1, return_lag_5, roll...",0.004100,0.006143,0.258751,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:35:10.484823,2026-07-25T11:35:10.732252,0.247404
3,Gradient Boosting,select_k_best,10,"rolling_return_20d, abs_return, squared_return...",0.004097,0.006155,0.255965,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:35:08.100072,2026-07-25T11:35:08.287845,0.187751
15,Gradient Boosting,random_forest_top_10,10,"rolling_abs_return_20d, risk_free_rate_decimal...",0.004208,0.006185,0.248732,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:35:11.314375,2026-07-25T11:35:11.534377,0.219980
13,Ridge Regression,random_forest_top_10,10,"rolling_abs_return_20d, risk_free_rate_decimal...",0.004082,0.006319,0.215751,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:35:10.759733,2026-07-25T11:35:10.771675,0.011925
12,Linear Regression,random_forest_top_10,10,"rolling_abs_return_20d, risk_free_rate_decimal...",0.004082,0.006321,0.215291,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:35:10.753336,2026-07-25T11:35:10.757904,0.004545
6,Random Forest,mutual_info_top_10,10,"rolling_volatility_20d, rolling_squared_return...",0.004348,0.006782,0.096716,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:35:08.330069,2026-07-25T11:35:08.855234,0.525145


# Final Feature Selection Comparison

In [43]:
all_feature_selection_results = pd.concat(
    [feature_set_results, selected_feature_results],
    ignore_index=True
)

In [44]:
all_feature_selection_results = all_feature_selection_results.sort_values("RMSE")

all_feature_selection_results

,model,feature_set,num_features,features,MAE,RMSE,R2,split_date,train_start_date,train_end_date,test_start_date,test_end_date,train_rows,test_rows,training_start_timestamp,training_end_timestamp,training_duration_seconds
8,Linear Regression,selected_macro,17,"daily_return, return_lag_1, return_lag_5, roll...",0.004002,0.005929,0.309563,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:27:18.358863,2026-07-25T11:27:18.367857,0.008966
9,Ridge Regression,selected_macro,17,"daily_return, return_lag_1, return_lag_5, roll...",0.004004,0.005930,0.309471,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:27:18.370337,2026-07-25T11:27:18.395778,0.025417
12,Linear Regression,select_k_best,10,"rolling_return_20d, abs_return, squared_return...",0.003992,0.005971,0.299858,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:35:07.344614,2026-07-25T11:35:07.349058,0.004415
13,Ridge Regression,select_k_best,10,"rolling_return_20d, abs_return, squared_return...",0.003994,0.005971,0.299840,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:35:07.350981,2026-07-25T11:35:07.362160,0.011162
14,Random Forest,select_k_best,10,"rolling_return_20d, abs_return, squared_return...",0.004017,0.005971,0.299816,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:35:07.364282,2026-07-25T11:35:08.071649,0.707354
0,Linear Regression,market_only,12,"daily_return, return_lag_1, return_lag_5, roll...",0.004044,0.005985,0.296477,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:27:13.876752,2026-07-25T11:27:13.903517,0.026739
1,Ridge Regression,market_only,12,"daily_return, return_lag_1, return_lag_5, roll...",0.004046,0.005986,0.296300,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:27:13.906009,2026-07-25T11:27:13.927916,0.021889
7,Gradient Boosting,full_macro,21,"daily_return, return_lag_1, return_lag_5, roll...",0.004095,0.006003,0.292264,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:27:18.047949,2026-07-25T11:27:18.331032,0.283060
19,Gradient Boosting,mutual_info_top_10,10,"rolling_volatility_20d, rolling_squared_return...",0.004192,0.006074,0.275508,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:35:08.882660,2026-07-25T11:35:09.066777,0.184096
3,Gradient Boosting,market_only,12,"daily_return, return_lag_1, return_lag_5, roll...",0.004041,0.006086,0.272456,2024-01-01,2018-01-31,2023-12-29,2024-01-02,2025-12-01,31269,10101,2026-07-25T11:27:14.989958,2026-07-25T11:27:16.651270,1.661288


# Save Output

In [45]:
all_feature_selection_results.to_csv(
    OUTPUT_PATH / "feature_selection_model_comparison.csv",
    index=False
)

mutual_info_ranking.to_csv(
    OUTPUT_PATH / "mutual_info_feature_ranking.csv",
    index=False
)

rf_feature_importance.to_csv(
    OUTPUT_PATH / "random_forest_feature_importance.csv",
    index=False
)

print("Saved feature selection outputs.")

Saved feature selection outputs.


# Interpretation

The feature selection experiments compare whether all macro features should be included or whether a smaller selected feature set performs better.

The main question is whether the expanded FRED macro data improves prediction after feature selection. If the selected feature sets perform better than the full macro feature set, that means adding every macro column at once created noise or redundancy. If the full macro feature set still performs best, then the models are benefiting from the broader macro context.

The final model should use the feature set with the strongest out-of-sample RMSE and R2, while also considering whether the selected features make sense for the project story.